# Checking Constraints in pyRDDLGym

In this notebook, we show how to check constraints on states or actions during simulation in pyRDDLGym.

First, install the required packages:

In [1]:
pip install --quiet --upgrade pip pyRDDLGym rddlrepository

Note: you may need to restart the kernel to use updated packages.


Import the required packages:

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pyRDDLGym
from pyRDDLGym.core.policy import RandomAgent

Let's simulate the elevators domain, with constraint checking on actions:

In [2]:
env = pyRDDLGym.make('Elevators', '0', enforce_action_constraints=True)
agent = RandomAgent(env.action_space, num_actions=env.max_allowed_actions)
agent.evaluate(env, episodes=1, verbose=False, render=False)

RDDLActionPreconditionNotSatisfiedError: Precondition 0 is not satisfied for actions {'move-current-dir': array([False, False]), 'open-door': array([ True, False]), 'close-door': array([ True, False])}.
>> ( forall_{?e: elevator} [ ( ( open-door(?e) + close-door(?e) ) + move-current-dir(?e) ) <= 1 ] )

As you can see, the simulation terminates with an invalid action, since the built-in random policy cannot account for arbitrary action-preconditions during simulation. Rejection sampling is a simple way to enforce constraints, at the expense of some extra computation:

In [3]:
def rejection_sample_action(state):
    action = agent.sample_action(state)
    sim_action = env.sampler.prepare_actions_for_sim(action)
    while not env.sampler.check_action_preconditions(sim_action, silent=True):
        action = agent.sample_action(state)
        sim_action = env.sampler.prepare_actions_for_sim(action)
    return action

Let's use this to sample actions that satisfy constraints:

In [4]:
state, _ = env.reset()
agent.reset()
total_reward = 0.0
for _ in range(env.horizon):
    action = rejection_sample_action(state)
    state, reward, term, trunc, _ = env.step(action)
    total_reward += reward
    if term or trunc: break
print(f'Total reward: {total_reward}')

Total reward: -1006.5


As you can see, sampling now proceeds without violating the action preconditions.